# PRACTICA: Transfer Learning y Fine Tuning con CNN

Clasificación de paisajes usando **MobileNetV2 preentrenado**.

Compararemos:
- Transfer Learning
- Fine Tuning


## Ejercicio 0 — Importación de librerías

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

## 1. Preparación de los datos

In [5]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_dirs = [
    'data/github_train_1',
    'data/github_train_2',
    'data/github_train_3'
]

test_dir = 'data/github_test'

In [6]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [7]:
# Cargar todos los datos de entrenamiento
train_generators = []

for path in train_dirs:
    gen = datagen.flow_from_directory(
        path,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )
    train_generators.append(gen)

val_generators = []

for path in train_dirs:
    gen = datagen.flow_from_directory(
        path,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )
    val_generators.append(gen)

Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


In [8]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 0 images belonging to 0 classes.


## 2. Cargar modelo preentrenado MobileNetV2

In [9]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## 3. Transfer Learning

In [11]:
num_classes = train_generators[0].num_classes

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)

predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

ValueError: Received an invalid value for `units`, expected a positive integer. Received: units=0

In [ ]:
history = model.fit(
    train_generators[0],
    validation_data=val_generators[0],
    epochs=5
)

### Evaluación Transfer Learning

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f'Accuracy en test: {test_acc}')

In [ ]:
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes

print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Matriz de confusión - Transfer Learning')
plt.show()

## 4. Fine Tuning

In [ ]:
# Descongelar últimas capas
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_finetune = model.fit(
    train_generators[0],
    validation_data=val_generators[0],
    epochs=5
)

### Evaluación Fine Tuning

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f'Accuracy en test tras fine tuning: {test_acc}')

In [ ]:
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes

print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Matriz de confusión - Fine Tuning')
plt.show()

## 5. Comparación de resultados

- La CNN hecha a mano suele necesitar más datos y más entrenamiento.
- **Transfer Learning** aprovecha características aprendidas en ImageNet.
- **Fine Tuning** mejora aún más ajustando las últimas capas del modelo.

En la mayoría de casos:

**Fine Tuning > Transfer Learning > CNN desde cero**